In [1]:
# STEP 1: Mount Google Drive & Download/Extract Dataset (robust)

import os, zipfile, shutil
from pathlib import Path
from google.colab import drive

# --- 1) Mount Drive ---
drive.mount('/content/drive')

# --- 2) Paths ---
download_url = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/ptz377bwb8-1.zip"
archive_path = Path("/content/PotatoDiseaseDataset.zip")
extract_path = Path("/content/PotatoDiseaseDataset")
dest_path = Path("/content/drive/MyDrive/Research/PotatoDiseaseDataset")

dest_path.parent.mkdir(parents=True, exist_ok=True)

# --- 3) Download & Extract (idempotent) ---
if not dest_path.exists():
    print("Downloading and extracting dataset to Drive (first run)...")
    if not archive_path.exists():
        os.system(f"wget -q -O {archive_path} {download_url}")
    # Clean any partial previous extract
    if extract_path.exists():
        shutil.rmtree(extract_path)
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(extract_path)
    # Move whole extracted tree into Drive folder
    shutil.move(str(extract_path), str(dest_path))
    print("✅ Dataset downloaded, extracted, and moved to Google Drive.")
else:
    print(f"✅ Destination path '{dest_path}' already exists. Skipping download/extract.")

# --- 4) Locate the actual dataset root dynamically ---
# Expected class folders:
expected = {"bacteria", "fungi", "healthy leaves", "nematodes", "pests", "phytophthora", "viruses"}

def find_dataset_root(base: Path, expected_classes: set):
    # walk and pick the first directory whose immediate subdirs intersect with expected classes
    for root, dirs, files in os.walk(base):
        dirset = set(dirs)
        if len(dirset & expected_classes) >= 5:   # at least 5 matches to be confident
            return Path(root)
    # fallback: common name used by this dataset
    fallback = base / "Potato Leaf Disease Dataset in Uncontrolled Environment"
    return fallback if fallback.exists() else None

data_dir = find_dataset_root(dest_path, expected)

if data_dir is None or not data_dir.exists():
    raise FileNotFoundError(
        f"Could not locate dataset root under {dest_path}. "
        "Please check the extracted folder name."
    )

print(f"\n Dataset root: {data_dir}")

# --- 5) Print class stats ---
def is_img(fname: str):
    return fname.lower().endswith((".jpg", ".jpeg", ".png"))

class_names = sorted([d for d in os.listdir(data_dir) if (data_dir/d).is_dir()])
print("\nClasses inside dataset folder:")
total_imgs = 0
for c in class_names:
    cdir = data_dir / c
    if not cdir.is_dir():
        continue
    n_imgs = sum(1 for f in os.listdir(cdir) if is_img(f))
    total_imgs += n_imgs
    print(f"{c:20}: {n_imgs} images")

print(f"\n📈 Total images in dataset: {total_imgs}\n")


Mounted at /content/drive
✅ Destination path '/content/drive/MyDrive/Research/PotatoDiseaseDataset' already exists. Skipping download/extract.

 Dataset root: /content/drive/MyDrive/Research/PotatoDiseaseDataset/Potato Leaf Disease Dataset in Uncontrolled Environment

Classes inside dataset folder:
Bacteria            : 569 images
Fungi               : 748 images
Healthy             : 201 images
Nematode            : 68 images
Pest                : 611 images
Phytopthora         : 347 images
Virus               : 532 images

📈 Total images in dataset: 3076



In [20]:
# --- Core Libraries ---
import os
import shutil
import numpy as np
import random
import sys
import time
import traceback
from pathlib import Path

# --- PyTorch and Data ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import transforms, datasets, models

# --- Models and Augmentations ---
import timm
from timm.loss import LabelSmoothingCrossEntropy
from timm.data import Mixup

# --- Evaluation ---
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Utilities ---
import pandas as pd
from tqdm import tqdm
from PIL import Image
import logging
from collections import Counter

from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
import torch.nn.functional as F

# CUDA check and device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("All libraries imported successfully!")
print("Torch version:", torch.__version__)
print("Using device:", device)

All libraries imported successfully!
Torch version: 2.8.0+cu126
Using device: cuda


In [3]:
# --- SafeImageFolder (robust dataset loader) ---
class SafeImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)
        self.invalid_files = []
    def __getitem__(self, index):
        path, target = self.samples[index]
        try:
            sample = self.loader(path)
            if self.transform is not None:
                sample = self.transform(sample)
            return sample, target
        except Exception as e:
            print(f"Error loading {path}: {e}")
            self.invalid_files.append(path)
            dummy = torch.zeros(3, 224, 224) if self.transform is None else self.transform(Image.new('RGB', (224, 224)))
            return dummy, -1
    def __len__(self):
        return len(self.samples)

# --- Dataset Validation ---
def validate_dataset(data_dir, class_names):
    print("Validating dataset...")
    total_images = 0
    for class_name in class_names:
        class_path = os.path.join(data_dir, class_name)
        if not os.path.exists(class_path):
            print(f"Warning: Class directory {class_path} does not exist")
            continue
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        print(f"{class_name}: {len(files)} images found")
        total_images += len(files)
        for f in files:
            file_path = os.path.join(class_path, f)
            try:
                Image.open(file_path).verify()
            except Exception as e:
                print(f"Error: Corrupted image {file_path}: {e}")
    print(f"Dataset validation complete: {total_images} images found.")

# --- Class Weights (for loss and imbalance) ---
def compute_class_weights(dataset, num_classes):
    class_counts = [0] * num_classes
    for _, label in dataset:
        if label != -1:
            class_counts[label] += 1
    class_weights = [0.] * num_classes
    total = float(sum(class_counts))
    for i in range(num_classes):
        if class_counts[i] > 0:
            class_weights[i] = total / class_counts[i]
        else:
            class_weights[i] = 0.
    return torch.tensor(class_weights, dtype=torch.float32), class_counts

# --- Weighted Random Sampler for Imbalance ---
def get_sampler(dataset, num_classes, num_samples):
    targets = []
    for _, label in dataset:
        if label != -1:
            targets.append(label)
    class_sample_count = Counter(targets)
    weights = [1.0 / class_sample_count[t] for t in targets]
    sampler = WeightedRandomSampler(weights, num_samples, replacement=True)
    return sampler

# --- Logger Setup ---
def setup_logging(variant, output_dir):
    log_file = os.path.join(output_dir, f'training_{variant}.log')
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(file_handler)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(console_handler)
    return logger

In [4]:
# # --- Weighted Label Smoothing Loss (for reference, if NOT using MixUp) ---
# class WeightedLabelSmoothingCrossEntropy(nn.Module):
#     def __init__(self, weight=None, smoothing=0.1):
#         super().__init__()
#         self.smoothing = smoothing
#         self.weight = weight  # tensor of shape (num_classes,) or None
#     def forward(self, input, target):
#         logprobs = torch.log_softmax(input, dim=-1)
#         n_classes = input.size(-1)
#         true_dist = torch.zeros_like(logprobs)
#         true_dist.fill_(self.smoothing / (n_classes - 1))
#         true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
#         if self.weight is not None:
#             weight = self.weight.unsqueeze(0)
#             loss = - (true_dist * logprobs * weight).sum(dim=1)
#         else:
#             loss = - (true_dist * logprobs).sum(dim=1)
#         return loss.mean()

In [5]:
# More aggressive augmentations can further help DenseNet regularization
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(40),   # slightly higher rotation range
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.15),
    transforms.RandomAffine(0, translate=(0.12, 0.12)),  # increase translate
    transforms.GaussianBlur(3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset (no transform yet)
dataset = SafeImageFolder(data_dir, transform=None)
class_names = dataset.classes
print(f"\nClass names: {class_names}")

# Print images per class (imbalance check)
cls_counts = [0] * len(class_names)
for _, label in dataset.samples:
    if label != -1:
        cls_counts[label] += 1
print("Image count per class:")
for cls, count in zip(class_names, cls_counts):
    print(f"{cls:15s} : {count}")

# Validate image files (detect corrupted)
validate_dataset(data_dir, class_names)

# Split (80/10/10)
train_ratio, val_ratio, test_ratio = 0.8, 0.1, 0.1
total_size = len(dataset)
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size
print(f"\nDataset split - Train: {train_size}, Val: {val_size}, Test: {test_size}")

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform
test_dataset.dataset.transform = val_transform

# Compute class weights (for loss)
class_weights, class_counts = compute_class_weights(train_dataset, len(class_names))
print("\nTrain set class counts:", class_counts)
print("Class weights (for loss):", class_weights.tolist())

# Weighted sampler (over/undersample to balance minibatches)
sampler = get_sampler(train_dataset, len(class_names), len(train_dataset))

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nDataLoader setup complete!")
print(f"Number of train batches: {len(train_loader)}")
print(f"Number of val batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")



Class names: ['Bacteria', 'Fungi', 'Healthy', 'Nematode', 'Pest', 'Phytopthora', 'Virus']
Image count per class:
Bacteria        : 569
Fungi           : 748
Healthy         : 201
Nematode        : 68
Pest            : 611
Phytopthora     : 347
Virus           : 532
Validating dataset...
Bacteria: 569 images found
Fungi: 748 images found
Healthy: 201 images found
Nematode: 68 images found
Pest: 611 images found
Phytopthora: 347 images found
Virus: 532 images found
Dataset validation complete: 3076 images found.

Dataset split - Train: 2460, Val: 307, Test: 309

Train set class counts: [461, 599, 165, 49, 488, 277, 421]
Class weights (for loss): [5.336225509643555, 4.106844902038574, 14.909090995788574, 50.20408248901367, 5.0409836769104, 8.880866050720215, 5.843230247497559]

DataLoader setup complete!
Number of train batches: 77
Number of val batches: 10
Number of test batches: 10


In [6]:
# # ---- MixUp function ----
# mixup_fn = Mixup(
#     mixup_alpha=0.2,
#     cutmix_alpha=1.0,
#     label_smoothing=0.1,
#     num_classes=len(class_names)
# )

In [7]:
# NOTE: pretrained=True loads ImageNet weights initially,
# but these are overwritten by your KD-trained checkpoint (efficientnetb0_kd_best.pt).

# # --- Student Model: EfficientNet-B0 (timm-based) ---
# class EfficientNetB0Custom(nn.Module):
#     def __init__(self, num_classes=7, dropout=0.2):
#         super().__init__()
#         self.base = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0, drop_rate=dropout)
#         self.pool = nn.AdaptiveAvgPool2d(1)
#         self.dropout = nn.Dropout(dropout)
#         self.head = nn.Linear(self.base.num_features, num_classes)  # 1280 features for efficientnet_b0

#     def forward(self, x):
#         x = self.base.forward_features(x)
#         x = self.pool(x).flatten(1)
#         x = self.dropout(x)
#         x = self.head(x)
#         return x

# ---- DEVICE ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
class_counts = {
    "Bacteria": 569,
    "Fungi": 748,
    "Healthy": 201,
    "Nematode": 68,
    "Pest": 611,
    "Phytopthora": 347,
    "Virus": 532
}


In [9]:
# # ---- Weighted Random Sampler Setup (for train loader) ----
# def make_weighted_sampler(labels):
#     from collections import Counter
#     counts = Counter(labels)
#     total_count = sum(counts.values())
#     class_weights = {cls: total_count / count for cls, count in counts.items()}
#     sample_weights = [class_weights[label] for label in labels]
#     sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
#     return sampler

# ---- Model Size Utility ----
def get_model_sizes(model, path_prefix="temp_model"):
    # --- weights only ---
    weights_path = f"{path_prefix}_weights.pth"
    torch.save(model.state_dict(), weights_path)
    weights_size = os.path.getsize(weights_path) / 1e6
    os.remove(weights_path)

    # --- full model (architecture + weights) ---
    full_path = f"{path_prefix}_full.pth"
    torch.save(model, full_path)
    full_size = os.path.getsize(full_path) / 1e6
    os.remove(full_path)

    return weights_size, full_size

# # ---- Inference Speed Utility ----
# def measure_inference_speed(model, device, input_size=(1, 3, 224, 224), runs=50):
#     import time
#     model.eval()
#     dummy = torch.randn(*input_size).to(device)
#     for _ in range(10): _ = model(dummy)  # Warm-up
#     start = time.time()
#     for _ in range(runs): _ = model(dummy)
#     end = time.time()
#     avg_time = (end - start) / runs * 1000  # ms/img
#     return avg_time

In [10]:

# Define and create a dedicated output directory for the student model
Output_dir_Student = os.path.join("outputs", "Output_dir_Student")
os.makedirs(Output_dir_Student, exist_ok=True)

In [11]:
# ======================
# Google Drive utilities
# ======================
IN_COLAB = 'google.colab' in sys.modules
DRIVE_MOUNT_POINT = '/content/drive'
DEFAULT_DRIVE_SUBDIR = 'MyDrive/ModelCheckpoints'

def _ensure_drive_mounted():
    """Mount Google Drive in Colab if available and not already mounted."""
    if IN_COLAB:
        from google.colab import drive  # noqa: F401
        if not os.path.ismount(DRIVE_MOUNT_POINT):
            drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

def _maybe_save_to_drive(local_path, drive_dir):
    """Copy a file to Google Drive, if drive_dir is provided and available."""
    if drive_dir is None:
        return
    try:
        os.makedirs(drive_dir, exist_ok=True)
        dst = os.path.join(drive_dir, os.path.basename(local_path))
        shutil.copy2(local_path, dst)
        print(f"☁️  Copied to Google Drive: {dst}")
    except Exception as e:
        print(f"⚠️  Failed to copy to Google Drive: {e}")

In [12]:
# ======================
# EarlyStopping (w/ Drive mirror)
# ======================
class EarlyStopping:
    def __init__(self, patience=10, delta=0, verbose=True, path='checkpoint.pt', drive_mirror_dir=None):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.path = path
        self.drive_mirror_dir = drive_mirror_dir
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.best_loss:.4f} --> {val_loss:.4f}). Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.best_loss = val_loss
        # Mirror to Google Drive if requested
        _maybe_save_to_drive(self.path, self.drive_mirror_dir)

# ======================
# Mixup Utility
# ======================
def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# ======================
# KD Loss
# ======================
def kd_loss(student_logits, teacher_logits, targets, alpha=0.7, T=4.0):
    log_student_soft = F.log_softmax(student_logits / T, dim=1)
    teacher_soft = F.softmax(teacher_logits / T, dim=1)
    kd = F.kl_div(log_student_soft, teacher_soft, reduction='batchmean') * (T * T)
    ce = F.cross_entropy(student_logits, targets)
    return alpha * kd + (1 - alpha) * ce

# ======================
# Helpers you referenced later
# ======================
def get_model_size(model):
    """Rough size of parameters only, in MB (float32 assumed)."""
    params = sum(p.numel() for p in model.parameters())
    return params * 4 / (1024 * 1024)  # 4 bytes per float32

# @torch.inference_mode()
# def measure_inference_speed(model, device, input_shape=(1, 3, 224, 224), warmup=10, iters=50):
#     """Simple throughput timing (ms/img) on dummy data."""
#     model.eval().to(device)
#     x = torch.randn(*input_shape, device=device)
#     if device.type == 'cuda':
#         torch.cuda.synchronize()
#     # warmup
#     for _ in range(warmup):
#         _ = model(x)
#     if device.type == 'cuda':
#         torch.cuda.synchronize()
#     t0 = time.time()
#     for _ in range(iters):
#         _ = model(x)
#     if device.type == 'cuda':
#         torch.cuda.synchronize()
#     elapsed = time.time() - t0
#     return (elapsed / iters) * 1000.0  # ms per img


In [13]:
# NOTE: This KD training loop is useful if want to retrain the student.
# ======================
# Training Function
# ======================
def train_kd(student_model, teacher_model, train_loader, val_loader, optimizer, scheduler, num_epochs,
             device, early_stopping, variant, output_dir, alpha=0.7, T=4.0):

    student_model.to(device)
    teacher_model.to(device)
    teacher_model.eval()

    train_losses, val_losses = [], []
    val_accs, val_precisions, val_recalls, val_f1s = [], [], [], []
    epoch_summaries = []
    best_model_path = os.path.join(output_dir, f'{variant}_best.pt')

    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        seen = 0

        for images, labels in train_loader:
            valid_mask = labels != -1
            if not valid_mask.any():
                continue
            images, labels = images[valid_mask].to(device), labels[valid_mask].to(device)

            images, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.4)

            with torch.no_grad():
                teacher_logits = teacher_model(images)
            student_logits = student_model(images)

            loss = lam * kd_loss(student_logits, teacher_logits, targets_a, alpha, T) + \
                   (1 - lam) * kd_loss(student_logits, teacher_logits, targets_b, alpha, T)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            bs = images.size(0)
            running_loss += loss.item() * bs
            seen += bs

        # IMPORTANT: divide by number of *seen* samples, not full dataset size (masks!)
        train_loss = running_loss / max(seen, 1)
        train_losses.append(train_loss)

        # ===== Validation =====
        student_model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_preds, val_labels_ = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                valid_mask = labels != -1
                if not valid_mask.any():
                    continue
                images, labels = images[valid_mask].to(device), labels[valid_mask].to(device)

                teacher_logits = teacher_model(images)
                student_logits = student_model(images)
                loss = kd_loss(student_logits, teacher_logits, labels, alpha, T)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(student_logits, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                val_preds.extend(predicted.detach().cpu().numpy())
                val_labels_.extend(labels.detach().cpu().numpy())

        val_loss /= max(val_total, 1)
        val_acc = 100.0 * val_correct / max(val_total, 1)
        val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
            val_labels_, val_preds, average='weighted', zero_division=0
        )

        val_losses.append(val_loss)
        val_accs.append(val_acc)
        val_precisions.append(val_precision)
        val_recalls.append(val_recall)
        val_f1s.append(val_f1)

        epoch_summary = {
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_accuracy': val_acc,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_f1': val_f1
        }
        epoch_summaries.append(epoch_summary)
        print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Early stopping (+ auto mirror best to Drive via EarlyStopping)
        early_stopping(val_loss, student_model)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

        if scheduler is not None:
            scheduler.step()

    # Save CSV
    df = pd.DataFrame(epoch_summaries)
    csv_path = os.path.join(output_dir, f'epoch_summary_efficientnetb0_kd.csv')
    df.to_csv(csv_path, index=False)
    print(f"Saved epoch summaries to {csv_path}")

    # Reload best model
    student_model.load_state_dict(torch.load(early_stopping.path, map_location=device))
    print("\nTraining complete! Student model reloaded from best checkpoint.")

    # ===== PLOTS =====
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses)+1), train_losses, 'b-', label='Train Loss')
    plt.plot(range(1, len(val_losses)+1), val_losses, 'r-', label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f'loss_plot_efficientnetb0_kd.png'))
    plt.show(); plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(val_accs)+1), val_accs, 'r-', label='Val Accuracy')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f'accuracy_plot_efficientnetb0_kd.png'))
    plt.show(); plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(val_precisions)+1), val_precisions, 'b-', label='Val Precision')
    plt.plot(range(1, len(val_recalls)+1), val_recalls, 'r-', label='Val Recall')
    plt.plot(range(1, len(val_f1s)+1), val_f1s, 'g-', label='Val F1')
    plt.title('Validation Precision, Recall, and F1-Score')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f'prf1_plot_efficientnetb0_kd.png'))
    plt.show(); plt.close()

    cm = confusion_matrix(val_labels_, val_preds, labels=range(len(class_names)))
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig(os.path.join(output_dir, f'confusion_matrix_efficientnetb0_kd.png'))
    plt.show(); plt.close()

    return student_model


In [14]:
# ======================
# Setup
# ======================
Output_dir_Student = os.path.join("outputs", "Output_dir_Student")
os.makedirs(Output_dir_Student, exist_ok=True)

def setup_logging(variant, output_dir):
    log_file = os.path.join(output_dir, f'training_{variant}.log')
    logger = logging.getLogger(variant)
    logger.setLevel(logging.INFO)
    # reset handlers only for this logger
    if logger.handlers:
        for h in logger.handlers[:]:
            logger.removeHandler(h)
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    return logger

variant = 'efficientnetb0_kd'
logger = setup_logging(variant, Output_dir_Student)


In [15]:
# ======================
# Google Drive target path (auto mirror best + final)
# ======================
DRIVE_DIR = None
if IN_COLAB:
    _ensure_drive_mounted()
    DRIVE_DIR = os.path.join(DRIVE_MOUNT_POINT, DEFAULT_DRIVE_SUBDIR, variant)
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"🔗 Google Drive mirroring enabled → {DRIVE_DIR}")


🔗 Google Drive mirroring enabled → /content/drive/MyDrive/ModelCheckpoints/efficientnetb0_kd


In [16]:
!pip install thop

In [17]:
# ==== Robust loader for student_model (EffNet-B0) & teacher_model (ViT-B/16) ====
import torchvision.models as tvm
import re

try:
    import timm
except Exception as e:
    raise RuntimeError("timm is required for the ViT teacher: pip install timm") from e

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Fill your Drive paths here ---
STUDENT_CKPT = "/content/drive/MyDrive/Research/efficientnetbo_kd_best.pt"
TEACHER_CKPT = "/content/drive/MyDrive/Research/vit_vit_base_patch16_224_best.pt"

# Optional: set to an int to override the inferred class count (e.g., 6).
# Leave as None to auto-match each checkpoint.
FORCE_STUDENT_NUM_CLASSES = None
FORCE_TEACHER_NUM_CLASSES = None

def _extract_state_dict(obj):
    """Handle Module / state_dict / nested dict formats; strip DP 'module.' prefix."""
    if isinstance(obj, nn.Module):
        sd = obj.state_dict()
    elif isinstance(obj, dict):
        if "state_dict" in obj: sd = obj["state_dict"]
        elif "model_state_dict" in obj: sd = obj["model_state_dict"]
        else: sd = obj
    else:
        sd = obj
    # strip 'module.' prefix
    return { (k.replace("module.","") if k.startswith("module.") else k): v for k, v in sd.items() }

def _infer_num_classes_from_sd(sd, fallback=7):
    """
    Try common classifier keys: EfficientNet(tv): 'base.classifier.1.weight'
                               EfficientNet(timm): 'classifier.weight'
                               ViT(timm): 'head.weight'
    Returns out_features (num_classes).
    """
    for key in [
        "base.classifier.1.weight",   # torchvision EfficientNet-B0 wrapper used in your code
        "classifier.1.weight",        # alt tv key
        "classifier.weight",          # timm effnet style
        "head.weight",                # timm ViT head
        "fc.weight",                  # some heads
    ]:
        if key in sd and sd[key].ndim == 2:
            return sd[key].shape[0]
    # fallback: search any (C_out, C_in) that looks like a final linear with small C_out
    for k,v in sd.items():
        if v.ndim == 2 and v.shape[0] <= 1000 and re.search(r"(head|class|fc)", k):
            return v.shape[0]
    return fallback

# -------- Build student (torchvision EfficientNet-B0 wrapper) ----------
class StudentEffB0_TV(nn.Module):
    def __init__(self, num_classes:int):
        super().__init__()
        m = tvm.efficientnet_b0(weights=None)
        in_feats = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_feats, num_classes)
        self.base = m
    def forward(self, x): return self.base(x)

def load_student(path, force_num_classes=None, device=DEVICE):
    obj = torch.load(path, map_location="cpu")
    sd  = _extract_state_dict(obj)
    ncls = force_num_classes if force_num_classes is not None else _infer_num_classes_from_sd(sd, fallback=7)
    model = StudentEffB0_TV(ncls).to(device).eval()
    # if forced classes differ from checkpoint, drop classifier weights from sd
    head_keys = ["base.classifier.1.weight","base.classifier.1.bias","classifier.1.weight","classifier.1.bias","classifier.weight","classifier.bias"]
    if force_num_classes is not None:
        for hk in head_keys:
            sd.pop(hk, None)
    miss, unexp = model.load_state_dict(sd, strict=False)
    print(f"[Student] built with num_classes={ncls} | loaded from {os.path.basename(path)}")
    if miss:  print("  missing keys (first 5):", list(miss)[:5])
    if unexp: print("  unexpected keys (first 5):", list(unexp)[:5])
    return model

# -------- Build teacher (timm ViT-B/16 224) ----------
def load_teacher(path, force_num_classes=None, device=DEVICE):
    obj = torch.load(path, map_location="cpu")
    sd  = _extract_state_dict(obj)
    ncls = force_num_classes if force_num_classes is not None else _infer_num_classes_from_sd(sd, fallback=7)
    model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=ncls).to(device).eval()
    # if forced classes differ, drop head weights from sd so load_state_dict succeeds
    if force_num_classes is not None:
        for hk in ["head.weight","head.bias","fc.weight","fc.bias"]:
            sd.pop(hk, None)
    miss, unexp = model.load_state_dict(sd, strict=False)
    print(f"[Teacher] built with num_classes={ncls} | loaded from {os.path.basename(path)}")
    if miss:  print("  missing keys (first 5):", list(miss)[:5])
    if unexp: print("  unexpected keys (first 5):", list(unexp)[:5])
    return model

# ---- Load them
student_model = load_student(STUDENT_CKPT, force_num_classes=FORCE_STUDENT_NUM_CLASSES, device=DEVICE)
teacher_model = load_teacher(TEACHER_CKPT, force_num_classes=FORCE_TEACHER_NUM_CLASSES, device=DEVICE)

# quick sanity
with torch.inference_mode():
    _ = student_model(torch.randn(1,3,224,224, device=DEVICE))
    _ = teacher_model(torch.randn(1,3,224,224, device=DEVICE))
print("✅ student_model & teacher_model are ready.")


[Student] built with num_classes=7 | loaded from efficientnetbo_kd_best.pt
[Teacher] built with num_classes=7 | loaded from vit_vit_base_patch16_224_best.pt
✅ student_model & teacher_model are ready.


In [18]:
# ==========================================================
# Stage 2: FP16 Fine-tuning + Export (Improved + Reports)
# ==========================================================
import os, copy, glob, time
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from torch import nn
import torch.nn.functional as F
from thop import profile  # for FLOPs

# -------------------------
# Configurations
# -------------------------
Output_dir_Student = "/content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16"
os.makedirs(Output_dir_Student, exist_ok=True)

fp32_ckpt_path = "/content/drive/MyDrive/Research/efficientnetbo_kd_best.pt"


variant = "efficientnetb0_fp16"
example_shape = (1, 3, 224, 224)
fp16_num_epochs = 15               # 15 epochs as requested
fp16_lr = 5e-5                     # small LR for stable fine-tuning
fp16_weight_decay = 0.01
early_stopping_patience = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()  # AMP works on CUDA; on CPU we fall back to FP32

torch.manual_seed(42)
np.random.seed(42)

# -------------------------
# Sanity checks
# -------------------------
required_names = ["student_model", "teacher_model", "train_loader", "val_loader", "test_loader"]
missing = [n for n in required_names if n not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects: {missing}")

student_model = globals()["student_model"]
teacher_model = globals()["teacher_model"].to(device).eval()
train_loader = globals()["train_loader"]
val_loader = globals()["val_loader"]
test_loader = globals()["test_loader"]

# -------------------------
# Load FP32 checkpoint
# -------------------------
sd = torch.load(fp32_ckpt_path, map_location=device)
if isinstance(sd, dict) and any(k in sd for k in ("state_dict", "model_state_dict")):
    sd = sd.get("state_dict", sd.get("model_state_dict"))
student_model.load_state_dict(sd)
student_model = student_model.to(device).eval()
print(f" Loaded FP32 KD checkpoint from: {fp32_ckpt_path}")

# # -------------------------
# # KD loss (same as before)
# # -------------------------
# def kd_loss(student_logits, teacher_logits, targets, alpha=0.7, T=4.0):
#     log_student_soft = F.log_softmax(student_logits / T, dim=1)
#     teacher_soft = F.softmax(teacher_logits / T, dim=1)
#     kd = F.kl_div(log_student_soft, teacher_soft, reduction="batchmean") * (T * T)
#     ce = F.cross_entropy(student_logits, targets)
#     return alpha * kd + (1 - alpha) * ce

# -------------------------
# Optimizer & Scheduler
# -------------------------
model_fp16_train = copy.deepcopy(student_model)  # keep original intact
optimizer = torch.optim.AdamW((p for p in model_fp16_train.parameters() if p.requires_grad),
                              lr=fp16_lr, weight_decay=fp16_weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=fp16_num_epochs)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_val_loss = float("inf")
epochs_no_improve = 0
ckpt_state_dict_path = os.path.join(Output_dir_Student, f"{variant}_best_fp32_state.pth")  # keep fp32 state pre-cast
history = {"train_losses": [], "val_losses": [], "val_accs": []}

# -------------------------
# Training loop with early stopping (AMP on CUDA)
# -------------------------
for epoch in range(fp16_num_epochs):
    model_fp16_train.train()
    running_loss, seen = 0.0, 0

    for images, labels in train_loader:
        if labels is None:
            continue
        valid_mask = labels != -1 if torch.is_tensor(labels) else None
        if valid_mask is not None:
            if not valid_mask.any():
                continue
            images, labels = images[valid_mask], labels[valid_mask]
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.no_grad():
            teacher_logits = teacher_model(images)

        if use_amp:
            with torch.cuda.amp.autocast():
                student_logits = model_fp16_train(images)
                loss = kd_loss(student_logits, teacher_logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            student_logits = model_fp16_train(images)
            loss = kd_loss(student_logits, teacher_logits, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        seen += images.size(0)

    train_loss = running_loss / max(seen, 1)
    history["train_losses"].append(train_loss)

    # Validation
    model_fp16_train.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            if labels is None:
                continue
            valid_mask = labels != -1 if torch.is_tensor(labels) else None
            if valid_mask is not None:
                if not valid_mask.any():
                    continue
                images, labels = images[valid_mask], labels[valid_mask]
            images, labels = images.to(device), labels.to(device)

            if use_amp:
                with torch.cuda.amp.autocast():
                    teacher_logits = teacher_model(images)
                    student_logits = model_fp16_train(images)
                    loss = kd_loss(student_logits, teacher_logits, labels)
            else:
                teacher_logits = teacher_model(images)
                student_logits = model_fp16_train(images)
                loss = kd_loss(student_logits, teacher_logits, labels)

            val_loss += loss.item() * images.size(0)
            preds = student_logits.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / max(val_total, 1)
    val_acc = 100.0 * val_correct / max(val_total, 1)
    history["val_losses"].append(val_loss)
    history["val_accs"].append(val_acc)

    print(f"Epoch {epoch+1}/{fp16_num_epochs} | Train {train_loss:.4f} | Val {val_loss:.4f} | Acc {val_acc:.2f}%")

    # Early stopping check (save best FP32 state_dict for safety)
    if val_loss < best_val_loss:
        torch.save(model_fp16_train.state_dict(), ckpt_state_dict_path)
        best_val_loss = val_loss
        epochs_no_improve = 0
        print(f" Saved best fine-tuned state_dict to {ckpt_state_dict_path}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stopping_patience:
            print(" Early stopping triggered.")
            break

    scheduler.step()

# -------------------------
# Build FP16 inference model & save to Drive
# -------------------------
# Reload best weights (fp32), then cast to half for inference/export
infer_model = copy.deepcopy(student_model).to(device)
infer_model.load_state_dict(torch.load(ckpt_state_dict_path, map_location=device))
infer_model.eval()

# Cast to half only when CUDA is available; otherwise keep fp32 to avoid CPU half issues
if use_amp:
    infer_model.half()
    print(" Model cast to FP16 for inference (CUDA).")
else:
    print(" CUDA not available; keeping model in FP32 for CPU inference.")

# Save both state_dict and full model (disk size check uses full model)
fp16_state_path = os.path.join(Output_dir_Student, f"{variant}_state.pth")
fp16_full_path = os.path.join(Output_dir_Student, f"{variant}_full.pth")
torch.save(infer_model.state_dict(), fp16_state_path)
torch.save(infer_model.to("cpu"), fp16_full_path)  # save on CPU for portability
print(f" FP16/FP32 (device-dependent) model saved: {fp16_full_path}")

# -------------------------
# Evaluate on test set (with AMP autocast if CUDA)
# -------------------------
# Move back to device for eval
infer_model = infer_model.to(device).eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        if labels is None:
            continue
        valid_mask = labels != -1 if torch.is_tensor(labels) else None
        if valid_mask is not None:
            if not valid_mask.any():
                continue
            images, labels = images[valid_mask], labels[valid_mask]
        images, labels = images.to(device), labels.to(device)
        if images.ndim == 3:
            images = images.unsqueeze(0)

        if use_amp:
            with torch.cuda.amp.autocast():
                outputs = infer_model(images)
        else:
            outputs = infer_model(images)

        preds = outputs.argmax(1)
        test_preds.extend(preds.detach().cpu().numpy())
        test_labels.extend(labels.detach().cpu().numpy())

test_preds, test_labels = np.array(test_preds), np.array(test_labels)
test_acc = 100.0 * (test_preds == test_labels).mean()
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="weighted", zero_division=0)

print(f"\n FP16(Test on CUDA) / FP32(Test on CPU) | "
      f"Acc: {test_acc:.2f}% | Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | F1: {test_f1:.4f}")

# -------------------------
# Per-class report & confusion matrix
# -------------------------
if hasattr(train_loader.dataset, "classes"):
    class_names = list(getattr(train_loader.dataset, "classes"))
elif hasattr(train_loader.dataset, "class_to_idx"):
    class_names = [k for k, _ in sorted(train_loader.dataset.class_to_idx.items(), key=lambda kv: kv[1])]
else:
    ncls = int(test_labels.max() + 1) if len(test_labels) > 0 else 0
    class_names = [str(i) for i in range(ncls)]

report = classification_report(test_labels, test_preds, target_names=class_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()
report_path = os.path.join(Output_dir_Student, f"per_class_report_{variant}.csv")
report_df.to_csv(report_path, index=True)
print(" Per-class report saved:", report_path)

cm = confusion_matrix(test_labels, test_preds, labels=range(len(class_names)))
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Confusion Matrix (FP16/FP32 depending on device)")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right"); ax.set_yticklabels(class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i,j]}", ha="center", va="center", fontsize=8, color="black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
cm_path = os.path.join(Output_dir_Student, f"confusion_matrix_{variant}.png")
fig.savefig(cm_path, dpi=150)
plt.close(fig)
print("🖼️ Confusion matrix saved:", cm_path)

# -------------------------
# Loss curve
# -------------------------
fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.plot(history["train_losses"], label="Train Loss")
ax2.plot(history["val_losses"], label="Val Loss")
ax2.set_title("FP16 Fine-tune Loss")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend()
plt.tight_layout()
loss_path = os.path.join(Output_dir_Student, f"loss_curve_{variant}.png")
plt.savefig(loss_path, dpi=150)
plt.close(fig2)
print(" Loss curve saved:", loss_path)

# -------------------------
# Model size, FLOPs & Inference speed
# -------------------------
def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

# Sizes (estimates & disk)
fp32_params = count_params(student_model.cpu())
fp32_est_mb = fp32_params * 4 / 1024 / 1024  # 4 bytes/param

# Disk size of saved (device-dependent) fp16/32 model
fp16_size_mb = file_size_mb(fp16_full_path) if os.path.exists(fp16_full_path) else float("nan")

# FLOPs calculation (using thop on FP32 copy for consistency)
dummy_input = torch.randn(*example_shape)
flops, params = profile(copy.deepcopy(student_model).cpu(), inputs=(dummy_input,), verbose=False)

# Inference speed (avg ms/image) on device
def measure_inference_latency(model, loader, warmup_batches=5, measure_batches=20):
    model.eval()
    total_imgs, total_time = 0, 0.0
    done_batches = 0
    starter, ender = (torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)) if torch.cuda.is_available() else (None, None)

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            if images.ndim == 3:
                images = images.unsqueeze(0)

            bs = images.size(0)

            # Warmup
            if done_batches < warmup_batches:
                if use_amp:
                    with torch.cuda.amp.autocast():
                        _ = model(images)
                else:
                    _ = model(images)
                done_batches += 1
                continue

            # Measure
            if torch.cuda.is_available():
                torch.cuda.synchronize()
                starter.record()
                if use_amp:
                    with torch.cuda.amp.autocast():
                        _ = model(images)
                else:
                    _ = model(images)
                ender.record()
                torch.cuda.synchronize()
                elapsed_ms = starter.elapsed_time(ender)
            else:
                t0 = time.perf_counter()
                _ = model(images)
                elapsed_ms = (time.perf_counter() - t0) * 1000.0

            total_imgs += bs
            total_time += elapsed_ms
            done_batches += 1
            if (done_batches - warmup_batches) >= measure_batches:
                break

    if total_imgs == 0:
        return float("nan")
    return total_time / total_imgs  # ms per image


 Loaded FP32 KD checkpoint from: /content/drive/MyDrive/Research/efficientnetbo_kd_best.pt


/tmp/ipython-input-2856836951.py:76: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-2856836951.py:105: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-2856836951.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/15 | Train 0.3362 | Val 0.2426 | Acc 90.23%
 Saved best fine-tuned state_dict to /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/efficientnetb0_fp16_best_fp32_state.pth
Epoch 2/15 | Train 0.1582 | Val 0.2316 | Acc 90.55%
 Saved best fine-tuned state_dict to /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/efficientnetb0_fp16_best_fp32_state.pth
Epoch 3/15 | Train 0.1522 | Val 0.2249 | Acc 90.23%
 Saved best fine-tuned state_dict to /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/efficientnetb0_fp16_best_fp32_state.pth
Epoch 4/15 | Train 0.1428 | Val 0.2145 | Acc 90.55%
 Saved best fine-tuned state_dict to /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/efficientnetb0_fp16_best_fp32_state.pth
Epoch 5/15 | Train 0.1322 | Val 0.2215 | Acc 90.23%
Epoch 6/15 | Train 0.1283 | Val 0.2192 | Acc 89.90%
Epoch 7/15 | Train 0.1238 | Val 0.2149 | Acc 90.55%
Epoch 8/15 | Train 0.1252 | Val 0.2221 | Acc 91.

/tmp/ipython-input-2856836951.py:216: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



 FP16(Test on CUDA) / FP32(Test on CPU) | Acc: 89.64% | Precision: 0.9002 | Recall: 0.8964 | F1: 0.8970
 Per-class report saved: /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/per_class_report_efficientnetb0_fp16.csv
🖼️ Confusion matrix saved: /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/confusion_matrix_efficientnetb0_fp16.png
 Loss curve saved: /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/loss_curve_efficientnetb0_fp16.png


/tmp/ipython-input-2856836951.py:319: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-2856836951.py:331: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


NameError: name 'accuracy_score' is not defined

In [21]:
# =========================
# Final Clean Result Output
# =========================

# ---- Compute params & sizes ----
fp32_params = count_params(student_model.cpu())
fp32_est_mb = fp32_params * 4 / 1024 / 1024   # 4 bytes/param
fp16_size_mb = file_size_mb(fp16_full_path) if os.path.exists(fp16_full_path) else float("nan")

# ---- FLOPs (computed once, same for FP32/FP16) ----
dummy_input = torch.randn(*example_shape)
flops, _ = profile(copy.deepcopy(student_model).cpu(), inputs=(dummy_input,), verbose=False)

# ---- Latency: measure separately for FP32 & FP16 ----
fp32_latency = measure_inference_latency(student_model.to(device).float().eval(), test_loader)
fp16_latency = measure_inference_latency(infer_model.to(device).eval(), test_loader)

# ---- Accuracy / Precision / Recall / F1 (from test set preds) ----
acc = accuracy_score(test_labels, test_preds) * 100
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted', zero_division=0)

# ---- Save size report (text file) ----
size_report_path = os.path.join(Output_dir_Student, f"{variant}_size_report.txt")
with open(size_report_path, "w") as f:
    f.write("===== Model Efficiency Report =====\n")
    f.write(f"FP32 params: {fp32_params} ({fp32_params/1e6:.2f}M)\n")
    f.write(f"FP32 estimated size ≈ {fp32_est_mb:.2f} MB\n")
    f.write(f"Saved FP16 model size: {fp16_size_mb:.2f} MB\n")
    f.write(f"FLOPs (FP32 forward pass): {flops/1e6:.2f} MFLOPs\n")
    f.write(f"FP32 latency (ms/img): {fp32_latency:.2f}\n")
    f.write(f"FP16 latency (ms/img): {fp16_latency:.2f}\n")

# ---- Final Console Output ----
print("\n================= Final Results =================")
print(f"{'Model Variant':15} | {'Accuracy':8} | {'Precision':9} | {'Recall':7} | {'F1-score':8} | {'Params':7} | {'Size (MB)':9} | {'FLOPs':8} | {'Latency (ms/img)':17}")
print("-"*110)
print(f"{'FP32 (KD Student)':15} | {acc:.2f}%   | {precision:.3f}     | {recall:.3f}   | {f1:.3f}    | {fp32_params/1e6:.2f}M | {fp32_est_mb:.2f} MB  | {flops/1e6:.1f}M | {fp32_latency:.2f}")
print(f"{'FP16 (QAT Student)':15} | {acc:.2f}%   | {precision:.3f}     | {recall:.3f}   | {f1:.3f}    | {fp32_params/1e6:.2f}M | {fp16_size_mb:.2f} MB  | {flops/1e6:.1f}M | {fp16_latency:.2f}")
print("="*110)
print("Reports saved:")
print(f" - Per-class report: {report_path}")
print(f" - Confusion Matrix: {cm_path}")
print(f" - Loss Curve: {loss_path}")
print(f" - Size/FLOPs Report: {size_report_path}")

/tmp/ipython-input-2856836951.py:319: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-2856836951.py:331: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



================= Final Results =================
Model Variant   | Accuracy | Precision | Recall  | F1-score | Params  | Size (MB) | FLOPs    | Latency (ms/img) 
--------------------------------------------------------------------------------------------------------------
FP32 (KD Student) | 89.64%   | 0.900     | 0.896   | 0.897    | 4.02M | 15.32 MB  | 413.9M | 1.01
FP16 (QAT Student) | 89.64%   | 0.900     | 0.896   | 0.897    | 4.02M | 7.92 MB  | 413.9M | 1.08
Reports saved:
 - Per-class report: /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/per_class_report_efficientnetb0_fp16.csv
 - Confusion Matrix: /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/confusion_matrix_efficientnetb0_fp16.png
 - Loss Curve: /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/loss_curve_efficientnetb0_fp16.png
 - Size/FLOPs Report: /content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/efficientnetb0_fp16_size_report.txt


NOTE: This block re-loads the final saved QAT model from disk

In [19]:
# ==== Load the small model from Drive and evaluate on your test loader ====
import os, torch, torch.nn as nn
import torchvision.models as tvm
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# (Colab) mount Drive if needed
if "google.colab" in str(get_ipython()):
    from google.colab import drive
    drive.mount("/content/drive")

# >>>>>>> EDIT THIS <<<<<<<
SMALL_MODEL_PATH = "/content/drive/MyDrive/Research/Model_save/QAT/efficientnetb0_kd_fp16/efficientnetb0_fp16_state.pth"
NUM_CLASSES = len(class_names) if "class_names" in globals() else 7
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Minimal EffB0 wrapper (matches what you used)
class StudentEffB0_TV(nn.Module):
    def __init__(self, num_classes:int):
        super().__init__()
        m = tvm.efficientnet_b0(weights=None)
        in_feats = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_feats, num_classes)
        self.base = m
    def forward(self, x): return self.base(x)

def load_small_model(path: str, num_classes: int, device=DEVICE):
    """
    Tries, in order:
      1) torch.jit.load (TorchScript .pt)
      2) torch.load nn.Module (full saved model)
      3) torch.load state_dict (then build StudentEffB0_TV and load)
    Casts to FP32 on CPU (many ops don't support half on CPU).
    """
    # Case 1: TorchScript .pt
    try:
        mdl = torch.jit.load(path, map_location=device)
        mdl.eval()
        # If CPU, keep float32; if CUDA and the model is half, keep it
        if device.type == "cpu":
            try: mdl = mdl.float()
            except Exception: pass
        print(f"[OK] Loaded TorchScript model: {os.path.basename(path)}")
        return mdl.to(device).eval()
    except Exception:
        pass

    # Case 2/3: Eager model or state_dict
    obj = torch.load(path, map_location=device)
    if isinstance(obj, nn.Module):
        mdl = obj.to(device).eval()
        # ensure dtype is safe for CPU
        if device.type == "cpu":
            try: mdl = mdl.float()
            except Exception: pass
        print(f"[OK] Loaded full nn.Module: {os.path.basename(path)}")
        return mdl

    # Assume it's a state_dict
    sd = obj
    if isinstance(obj, dict) and any(k in obj for k in ("state_dict", "model_state_dict")):
        sd = obj.get("state_dict", obj.get("model_state_dict"))

    # strip DP prefix if any
    sd = { (k.replace("module.","") if k.startswith("module.") else k): v for k,v in sd.items() }

    mdl = StudentEffB0_TV(num_classes).to(device).eval()
    missing, unexpected = mdl.load_state_dict(sd, strict=False)
    if list(missing):   print("  missing keys (first 5):", list(missing)[:5])
    if list(unexpected):print("  unexpected keys (first 5):", list(unexpected)[:5])
    if device.type == "cpu":
        mdl = mdl.float()
    print(f"[OK] Built StudentEffB0_TV and loaded state_dict: {os.path.basename(path)}")
    return mdl

# NOTE: .ptl (Lite Interpreter) cannot be loaded in Python:
if SMALL_MODEL_PATH.endswith(".ptl"):
    raise RuntimeError("'.ptl' (Mobile Lite) models can't be loaded in Python. Export a TorchScript .pt or full/state_dict .pth to test here.")

small_model = load_small_model(SMALL_MODEL_PATH, NUM_CLASSES, device=DEVICE)

# ---- Evaluate ONLY this loaded model on your test loader ----
def collect_preds(model, loader, device=DEVICE):
    model.to(device).eval()
    ys, yh = [], []
    with torch.inference_mode():
        for xb, yb in loader:
            if yb is None:
                continue
            # drop invalid labels if present
            if torch.is_tensor(yb) and (yb == -1).any():
                mask = (yb != -1)
                if not mask.any():
                    continue
                xb, yb = xb[mask], yb[mask]
            xb, yb = xb.to(device), yb.to(device)
            # if model is half and CUDA, use autocast for safety
            use_amp = (device.type == "cuda")
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(xb)
            else:
                logits = model(xb)
            yh.append(logits.argmax(1).detach().cpu())
            ys.append(yb.detach().cpu())
    return torch.cat(ys).numpy(), torch.cat(yh).numpy()

y_true, y_pred = collect_preds(small_model, test_loader, device=DEVICE)

acc = accuracy_score(y_true, y_pred)
pM,rM,fM,_ = precision_recall_fscore_support(y_true, y_pred, average="macro",    zero_division=0)
pW,rW,fW,_ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

print(f"\n=== Test (ONLY the loaded small model) ===")
print(f"accuracy={acc:.4f} | f1_macro={fM:.4f} | precision_macro={pM:.4f} | recall_macro={rM:.4f}")
if 'class_names' in globals():
    print("\nPer-class report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# Optional: quick confusion matrix
try:
    import pandas as pd
    cm = confusion_matrix(y_true, y_pred, labels=range(int(max(y_true.max(), y_pred.max())+1)))
    if 'class_names' in globals() and len(class_names) == cm.shape[0]:
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    else:
        cm_df = pd.DataFrame(cm)
    print("\nConfusion matrix:\n", cm_df)
except Exception as e:
    print("Confusion matrix display skipped:", e)

# Small sanity prints: file size & model dtype
def file_size_mb(p):
    return os.path.getsize(p)/(1024*1024) if os.path.exists(p) else float('nan')
print(f"\nLoaded file: {SMALL_MODEL_PATH}  |  size: {file_size_mb(SMALL_MODEL_PATH):.2f} MB")

# Peek param dtype (if eager)
try:
    first_param = next(small_model.parameters())
    print("Model param dtype:", first_param.dtype, "| device:", first_param.device)
except StopIteration:
    print("TorchScript model: dtype printed via runtime (handled in eval).")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Built StudentEffB0_TV and loaded state_dict: efficientnetb0_fp16_state.pth


/tmp/ipython-input-3000727283.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



=== Test (ONLY the loaded small model) ===
accuracy=0.8964 | f1_macro=0.8943 | precision_macro=0.8880 | recall_macro=0.9043

Per-class report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        52
           1       0.89      0.83      0.86        76
           2       0.72      0.90      0.80        20
           3       0.90      0.90      0.90        10
           4       0.84      0.90      0.87        59
           5       0.94      0.94      0.94        35
           6       0.92      0.86      0.89        57

    accuracy                           0.90       309
   macro avg       0.89      0.90      0.89       309
weighted avg       0.90      0.90      0.90       309


Confusion matrix:
     0   1   2  3   4   5   6
0  52   0   0  0   0   0   0
1   0  63   1  0   9   2   1
2   0   0  18  0   0   0   2
3   0   0   0  9   1   0   0
4   0   4   0  1  53   0   1
5   0   2   0  0   0  33   0
6   0   2   6  0   0   0  49

Loaded